# House Price Prediction
**Author:** Srijan Chatterjee  
**Dataset:** Housing Prices Dataset — [mannatpruthi/house-prediction](https://www.kaggle.com/code/mannatpruthi/house-prediction)  
**Tech Stack:** Python, scikit-learn, Flask, Streamlit, pandas, matplotlib, seaborn

---
## Project Overview
This notebook covers the complete end-to-end Machine Learning pipeline for predicting house prices using the Housing Prices Dataset.

### Dataset Columns:
| Column | Type | Description |
|---|---|---|
| `area` | numeric | House area in square feet |
| `bedrooms` | numeric | Number of bedrooms |
| `bathrooms` | numeric | Number of bathrooms |
| `stories` | numeric | Number of floors |
| `mainroad` | yes/no | Connected to main road |
| `guestroom` | yes/no | Has a guest room |
| `basement` | yes/no | Has a basement |
| `hotwaterheating` | yes/no | Has hot water heating |
| `airconditioning` | yes/no | Has air conditioning |
| `parking` | numeric | Number of parking spaces |
| `prefarea` | yes/no | Located in preferred area |
| `furnishingstatus` | categorical | furnished / semi-furnished / unfurnished |
| `price` | numeric | **Target variable** — house sale price |

### Steps Covered:
1. Import Libraries
2. Load & Explore Dataset
3. Data Preprocessing (encode yes/no & furnishingstatus)
4. Exploratory Data Analysis (EDA)
5. Feature Engineering & Train/Val Split
6. Model Training (LinearRegression, Ridge, Lasso)
7. Model Evaluation
8. Save Model Artefacts
9. Load Model & Make Predictions
10. Flask API – Backend Code
11. Streamlit UI – Frontend Code

## 1. Import Libraries

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics         import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.style.use('seaborn-v0_8-darkgrid')

print('All libraries imported successfully!')

## 2. Load & Explore Dataset

In [ ]:
# Load the Housing Prices dataset
df = pd.read_csv('data/train.csv')

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Dataset Info
print('Dataset Info:')
print(f'  Total Rows    : {df.shape[0]:,}')
print(f'  Total Columns : {df.shape[1]}')
print(f'  Missing Values: {df.isnull().sum().sum()}')
print(f'\nColumn names:')
print(list(df.columns))

In [ ]:
# Statistical Summary
df.describe().round(2)

In [ ]:
# Target variable statistics
print('Price Statistics:')
print(f'  Mean   : {df["price"].mean():,.0f}')
print(f'  Median : {df["price"].median():,.0f}')
print(f'  Std    : {df["price"].std():,.0f}')
print(f'  Min    : {df["price"].min():,.0f}')
print(f'  Max    : {df["price"].max():,.0f}')

## 3. Data Preprocessing

In [ ]:
TARGET = 'price'

# Encode binary yes/no columns as 1/0
BINARY_COLS = ['mainroad', 'guestroom', 'basement', 'hotwaterheating',
               'airconditioning', 'prefarea']
for col in BINARY_COLS:
    df[col] = df[col].map({'yes': 1, 'no': 0})

# Encode furnishingstatus: furnished=2, semi-furnished=1, unfurnished=0
df['furnishingstatus'] = df['furnishingstatus'].map(
    {'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0}
)

print('After encoding – missing values per feature:')
print(df.isnull().sum())

In [ ]:
# Impute any remaining missing values with median
for col in df.columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print('After imputation – total missing values:', df.isnull().sum().sum())
print('Clean dataset shape:', df.shape)

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# --- 4.1  Price Distribution + Area scatter ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['price'], bins=40, color='#3b82d4', edgecolor='white')
axes[0].axvline(df['price'].mean(),   color='red',   linestyle='--',
                label=f'Mean: {df["price"].mean():,.0f}')
axes[0].axvline(df['price'].median(), color='green', linestyle='--',
                label=f'Median: {df["price"].median():,.0f}')
axes[0].set_title('Price Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].scatter(df['area'], df['price'],
                alpha=0.4, color='#7c5cd8', edgecolors='white', linewidths=0.3, s=20)
axes[1].set_title('Area vs Price', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Area (sq ft)')
axes[1].set_ylabel('Price')

plt.tight_layout()
plt.show()

In [ ]:
# --- 4.2  Bedrooms vs Avg Price  |  Furnishing Status vs Avg Price ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bed_mean = df.groupby('bedrooms')['price'].mean()
axes[0].bar(bed_mean.index, bed_mean.values, color='#7c5cd8', edgecolor='white')
axes[0].set_title('Bedrooms vs Avg Price', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Bedrooms')
axes[0].set_ylabel('Average Price')

# Re-read original for furnishingstatus display
df_raw = pd.read_csv('data/train.csv')
furn_mean = df_raw.groupby('furnishingstatus')['price'].mean().sort_values(ascending=False)
axes[1].bar(furn_mean.index, furn_mean.values,
            color=['#3b82d4', '#7c5cd8', '#22c55e'], edgecolor='white')
axes[1].set_title('Furnishing Status vs Avg Price', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Furnishing Status')
axes[1].set_ylabel('Average Price')

plt.tight_layout()
plt.show()

In [ ]:
# --- 4.3  Correlation Heatmap ---
plt.figure(figsize=(12, 8))
NUMERIC_FEATURES = [
    'area', 'bedrooms', 'bathrooms', 'stories',
    'mainroad', 'guestroom', 'basement', 'hotwaterheating',
    'airconditioning', 'parking', 'prefarea', 'furnishingstatus'
]
corr = df[NUMERIC_FEATURES + [TARGET]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 4.4  Top feature correlations with price ---
corr_target = df[NUMERIC_FEATURES + [TARGET]].corr()['price'].drop('price').sort_values(ascending=False)
print('Feature correlations with price:')
print(corr_target.to_string())

## 5. Feature Engineering & Train / Val Split

In [ ]:
X = df[NUMERIC_FEATURES]
y = df[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set   : {X_train.shape[0]} rows')
print(f'Validation set : {X_val.shape[0]} rows')
print(f'Features used  : {X_train.shape[1]}')
print(f'Feature names  : {list(X.columns)}')

## 6. Model Training

In [ ]:
# Define 3 candidate models inside sklearn Pipelines
def build_pipeline(estimator):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('model',  estimator)
    ])

candidates = {
    'LinearRegression': build_pipeline(LinearRegression()),
    'Ridge'           : build_pipeline(Ridge(alpha=10)),
    'Lasso'           : build_pipeline(Lasso(alpha=50, max_iter=10000)),
}

print('Model pipelines created:')
for name in candidates:
    print(f'  - {name}')

In [ ]:
# Train all models and compare
results = {}
best_name, best_pipe, best_rmse = None, None, float('inf')

print(f'{"Model":<22} {"RMSE":>14} {"MAE":>14} {"R2":>8}')
print('-' * 62)

for name, pipe in candidates.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_val)
    rmse  = np.sqrt(mean_squared_error(y_val, preds))
    mae   = mean_absolute_error(y_val, preds)
    r2    = r2_score(y_val, preds)
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    marker = ' <-- BEST' if rmse < best_rmse else ''
    print(f'{name:<22} {rmse:>14,.0f} {mae:>14,.0f} {r2:>8.4f}{marker}')
    if rmse < best_rmse:
        best_rmse, best_name, best_pipe = rmse, name, pipe

print(f'\nBest model: {best_name}  (RMSE = {best_rmse:,.0f})')

## 7. Model Evaluation

In [ ]:
# Evaluate best model
y_pred = best_pipe.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae  = mean_absolute_error(y_val, y_pred)
r2   = r2_score(y_val, y_pred)

print(f'Best Model     : {best_name}')
print(f'RMSE           : {rmse:,.2f}')
print(f'MAE            : {mae:,.2f}')
print(f'R2 Score       : {r2:.4f}  ({r2*100:.2f}% variance explained)')

In [ ]:
# --- 7.1  Actual vs Predicted  |  Residual Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_val, y_pred, alpha=0.5, color='#3b82d4', edgecolors='white', linewidths=0.3)
mn = min(y_val.min(), y_pred.min())
mx = max(y_val.max(), y_pred.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].set_title(f'Actual vs Predicted – {best_name}', fontsize=13, fontweight='bold')
axes[0].legend()

residuals = y_val - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, color='#7c5cd8', edgecolors='white', linewidths=0.3)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted Price')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# --- 7.2  Feature Importance (absolute coefficients) ---
coefs    = best_pipe.named_steps['model'].coef_
feat_imp = pd.Series(np.abs(coefs), index=NUMERIC_FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
feat_imp.plot(kind='barh', ax=ax, color='#7c5cd8', edgecolor='white')
ax.set_title('Feature Importance (|Coefficient|)', fontsize=13, fontweight='bold')
ax.set_xlabel('|Coefficient Value|')
plt.tight_layout()
plt.show()

print('\nTop 5 most important features:')
print(feat_imp.sort_values(ascending=False).head(5).to_string())

In [ ]:
# --- 7.3  Model Comparison Bar Chart (RMSE & R2) ---
model_names = list(results.keys())
rmse_values = [results[m]['RMSE'] for m in model_names]
r2_values   = [results[m]['R2']   for m in model_names]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#3b82d4', '#7c5cd8', '#22c55e']

bars = axes[0].bar(model_names, rmse_values, color=colors, edgecolor='white')
axes[0].set_title('RMSE Comparison (lower is better)', fontweight='bold')
axes[0].set_ylabel('RMSE')
axes[0].bar_label(bars, labels=[f'{v:,.0f}' for v in rmse_values], padding=4, fontsize=9)

bars2 = axes[1].bar(model_names, r2_values, color=colors, edgecolor='white')
axes[1].set_title('R2 Score Comparison (higher is better)', fontweight='bold')
axes[1].set_ylabel('R2 Score')
axes[1].set_ylim(0, 1.05)
axes[1].bar_label(bars2, labels=[f'{v:.4f}' for v in r2_values], padding=4, fontsize=9)

plt.tight_layout()
plt.show()

## 8. Save Model Artefacts

In [ ]:
os.makedirs('model', exist_ok=True)

MODEL_PATH    = 'model/house_price_model.pkl'
FEATURES_PATH = 'model/feature_columns.json'
METRICS_PATH  = 'model/metrics.json'

# Save trained pipeline
joblib.dump(best_pipe, MODEL_PATH)
print(f'Model saved      -> {MODEL_PATH}')

# Save feature list
with open(FEATURES_PATH, 'w') as f:
    json.dump(NUMERIC_FEATURES, f, indent=2)
print(f'Features saved   -> {FEATURES_PATH}')

# Save metrics
metrics_out = {
    'best_model': best_name,
    'RMSE' : round(results[best_name]['RMSE'], 2),
    'MAE'  : round(results[best_name]['MAE'],  2),
    'R2'   : round(results[best_name]['R2'],   4),
    'all_models': {
        k: {m: round(v, 2) for m, v in mv.items()}
        for k, mv in results.items()
    }
}
with open(METRICS_PATH, 'w') as f:
    json.dump(metrics_out, f, indent=2)
print(f'Metrics saved    -> {METRICS_PATH}')

## 9. Load Model & Make Predictions

In [ ]:
# Load model back and verify
loaded_model    = joblib.load(MODEL_PATH)
loaded_features = json.load(open(FEATURES_PATH))

print('Model loaded successfully!')
print(f'Features ({len(loaded_features)}): {loaded_features}')

In [ ]:
# Sample predictions on 5 houses
# Encoding: mainroad/guestroom/basement/hotwaterheating/airconditioning/prefarea -> yes=1, no=0
# furnishingstatus -> furnished=2, semi-furnished=1, unfurnished=0
#
# Columns: area, bedrooms, bathrooms, stories, mainroad, guestroom, basement,
#          hotwaterheating, airconditioning, parking, prefarea, furnishingstatus
sample_houses = pd.DataFrame([
    [7420, 4, 2, 3, 1, 0, 0, 0, 1, 2, 1, 2],   # Large furnished w/ AC
    [4000, 3, 1, 2, 1, 0, 1, 0, 0, 1, 0, 1],   # Mid-size semi-furnished
    [2000, 2, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],   # Small unfurnished
    [9600, 5, 3, 4, 1, 1, 1, 1, 1, 3, 1, 2],   # Luxury fully equipped
    [3500, 3, 1, 2, 1, 0, 0, 0, 1, 1, 0, 1],   # Average semi-furnished
], columns=loaded_features)

predictions = loaded_model.predict(sample_houses)

print('Sample House Price Predictions:')
print('-' * 60)
for i, (_, house) in enumerate(sample_houses.iterrows()):
    print(f'House {i+1}: Area={house["area"]:,} sqft | '
          f'Beds={int(house["bedrooms"])} | '
          f'AC={"Yes" if house["airconditioning"] else "No"} '
          f'-> Predicted: {predictions[i]:,.0f}')

## 10. Flask API – Backend (`backend/app.py`)

```
Run: python backend/app.py
Starts Flask REST API on http://localhost:5000

Endpoints:
  GET  /            -> health check
  GET  /model-info  -> feature list + saved metrics
  POST /predict     -> send feature JSON (encoded), get predicted_price back

Note: Send binary features as 0/1 and furnishingstatus as 0/1/2
```

In [ ]:
# Display backend/app.py source
with open('backend/app.py', 'r') as f:
    print(f.read())

## 11. Streamlit UI – Frontend (`frontend/ui.py`)

```
Run: streamlit run frontend/ui.py
Opens interactive web app on http://localhost:8501

Tabs:
  1. Predict        -> inputs for all 12 features, calls Flask API, shows price
  2. Dataset Explorer -> raw data, EDA charts, correlation heatmap
  3. Model Info     -> metrics, model comparison chart, saved training charts
```

In [ ]:
# Display frontend/ui.py source
with open('frontend/ui.py', 'r') as f:
    print(f.read())

## Summary

| Item | Detail |
|---|---|
| **Author** | Srijan Chatterjee |
| **Dataset** | Housing Prices Dataset (Kaggle) – 545 rows × 13 columns |
| **Dataset Source** | https://www.kaggle.com/code/mannatpruthi/house-prediction |
| **Features** | 12 features (5 numeric + 6 binary + 1 ordinal) |
| **Models Compared** | LinearRegression, Ridge (α=10), Lasso (α=50) |
| **Best Model** | LinearRegression (sklearn Pipeline) |
| **R² Score** | 0.9136 |
| **RMSE** | ~653,784 |
| **MAE** | ~532,353 |
| **Backend** | Flask REST API – `python backend/app.py` (port 5000) |
| **Frontend** | Streamlit Web App – `streamlit run frontend/ui.py` (port 8501) |
| **Model saved** | `model/house_price_model.pkl` |
| **Dataset link** | https://www.kaggle.com/code/mannatpruthi/house-prediction |